# [6.1] SAE Variants - Exercises

Implement the small mechanics behind ReLU/L1, TopK, Gated, and JumpReLU SAEs before trusting sparse feature claims on real model activations.

```yaml
gt_tier: GT-1 with GT-0 planted-feature controls
exercise_id: 6.1-sae-variants
expected_runtime: 60-90 minutes for CPU exercises; several minutes for CUDA Pythia SAE preflight
requires_gpu: true for the released-checkpoint preflight; false for the implementation exercises
```

Reading map: review toy superposition, sparse autoencoders, ROC AUC, and activation steering. Failure modes to watch for: TopK before ReLU, L0 over the wrong dimension, dense toy features, duplicate dictionary rows, threshold-only validation, and final-token steering accidentally applied to every position.


In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t
import torch.nn.functional as F

chapter = "chapter6_sparse_feature_methods"
section = "part1_sae_variants"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_sae_variants.tests as tests


@dataclass(frozen=True)
class ToySuperpositionBatch:
    feature_acts: t.Tensor
    activations: t.Tensor
    dictionary: t.Tensor


@dataclass(frozen=True)
class SAEVariantMetrics:
    name: str
    l0: float
    feature_density_mean: float
    dead_feature_fraction: float
    reconstruction_mse: float


@dataclass(frozen=True)
class DictionaryRecoveryReport:
    mean_best_cosine: float
    recovered_fraction: float
    duplicate_fraction: float
    best_learned_for_true: t.Tensor


@dataclass(frozen=True)
class FeatureAUCReport:
    feature_id: int
    auc: float


@dataclass(frozen=True)
class SteeringComparisonReport:
    baseline_mean: float
    steered_mean: float
    random_mean: float
    steered_delta: float
    random_delta: float
    passes_control: bool


## Encoder Variants

Difficulty: easy. Importance: high. Expected output: the test checks soft-threshold ReLU/L1, row-wise TopK, gated detection, and JumpReLU thresholding. Common bug: applying TopK before ReLU and preserving negative activations.


In [ ]:
def relu_l1_encode(pre_acts: t.Tensor, *, l1_coefficient: float = 0.0) -> t.Tensor:
    raise NotImplementedError()


def topk_encode(pre_acts: t.Tensor, *, k: int) -> t.Tensor:
    raise NotImplementedError()


def gated_encode(
    pre_acts: t.Tensor,
    gate_logits: t.Tensor,
    *,
    gate_threshold: float = 0.0,
) -> t.Tensor:
    raise NotImplementedError()


def jumprelu_encode(pre_acts: t.Tensor, *, threshold: float) -> t.Tensor:
    raise NotImplementedError()


tests.test_encoder_variants_match_reference_and_sparsity_rules(
    relu_l1_encode,
    topk_encode,
    gated_encode,
    jumprelu_encode,
)


## Reconstruction Metrics

Difficulty: medium. Importance: high. Expected output: the identity decoder has reconstruction MSE `0.0`, L0 `1.5`, and no dead features. Common bug: computing L0 over the whole batch instead of per activation vector.


In [ ]:
def decode_features(
    feature_acts: t.Tensor,
    decoder_weight: t.Tensor,
    decoder_bias: t.Tensor | None = None,
) -> t.Tensor:
    raise NotImplementedError()


def feature_density(feature_acts: t.Tensor, threshold: float = 0.0) -> t.Tensor:
    raise NotImplementedError()


def l0(feature_acts: t.Tensor, threshold: float = 0.0) -> float:
    raise NotImplementedError()


def dead_feature_fraction(feature_acts: t.Tensor, threshold: float = 0.0) -> float:
    raise NotImplementedError()


def sae_variant_metrics(
    name: str,
    *,
    activations: t.Tensor,
    reconstructed_activations: t.Tensor,
    feature_acts: t.Tensor,
) -> SAEVariantMetrics:
    raise NotImplementedError()


tests.test_decode_and_metrics_match_identity_contract(
    decode_features,
    sae_variant_metrics,
    feature_density,
    l0,
    dead_feature_fraction,
)


## Planted Features

Difficulty: medium. Importance: high. Expected output: the fixed-seed toy batch has shapes `(32, 6)`, `(32, 3)`, and `(6, 3)`, and dictionary rows have unit norm. Common bug: sampling dense latent features instead of sparse feature activations.


In [ ]:
def make_toy_superposition_batch(
    *,
    batch: int = 256,
    n_features: int = 8,
    d_model: int = 4,
    feature_probability: float = 0.2,
    noise_scale: float = 0.0,
    seed: int = 0,
) -> ToySuperpositionBatch:
    raise NotImplementedError()


def density_is_nondegenerate(
    feature_acts: t.Tensor,
    *,
    min_active_fraction: float = 0.05,
    max_active_fraction: float = 0.95,
) -> bool:
    raise NotImplementedError()


tests.test_toy_superposition_batch_has_planted_sparse_structure(
    make_toy_superposition_batch,
    density_is_nondegenerate,
)


## Dictionary Recovery

Difficulty: medium. Importance: high. Expected output: a duplicated decoder should recover `2/3` true directions and report duplicate fraction `1/3`. Common bug: measuring best true feature for each learned row rather than best learned row for each true feature.


In [ ]:
def dictionary_recovery_report(
    learned_decoder: t.Tensor,
    true_dictionary: t.Tensor,
    *,
    threshold: float = 0.8,
) -> DictionaryRecoveryReport:
    raise NotImplementedError()


tests.test_dictionary_recovery_detects_duplicates_and_missing_features(
    dictionary_recovery_report,
)


## Feature Validation

Difficulty: medium. Importance: high. Expected output: the best feature has polarity-corrected AUC `1.0`, and raw AUC distinguishes predictive from antipredictive features. Common bug: using one threshold's accuracy rather than rank-based ROC AUC.


In [ ]:
def roc_auc_binary(scores: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def best_feature_auc(feature_acts: t.Tensor, labels: t.Tensor) -> FeatureAUCReport:
    raise NotImplementedError()


tests.test_best_feature_auc_handles_predictive_and_antipredictive_features(
    roc_auc_binary,
    best_feature_auc,
)


## Decoder-Vector Steering

Difficulty: medium. Importance: high. Expected output: default steering changes only the final position, all-position steering changes every position, and the target decoder direction beats the random control. Common bug: applying final-token steering to every sequence position.


In [ ]:
def apply_decoder_steering(
    activations: t.Tensor,
    decoder_vectors: t.Tensor,
    feature_ids: t.Tensor | list[int],
    coefficients: t.Tensor | list[float] | float,
    *,
    positions: Literal["all", "last"] = "last",
) -> t.Tensor:
    raise NotImplementedError()


def steering_comparison_report(
    baseline_scores: t.Tensor,
    steered_scores: t.Tensor,
    random_control_scores: t.Tensor,
) -> SteeringComparisonReport:
    raise NotImplementedError()


tests.test_decoder_steering_changes_last_position_and_reports_control(
    apply_decoder_steering,
    steering_comparison_report,
)


## Final Verification

After the implementation cells pass, compare your functions against the reference implementation and then run the released-checkpoint preflight from a Python process with CUDA available:

```python
from part1_sae_variants import solutions
solutions.run_gpu_test(max_vram_gb=24.0)
```

The current checked path trains a tiny TopK SAE on pinned Pythia-70M hidden states and requires held-out reconstruction improvement, nondegenerate sparse density, feature AUC, a permuted-decoder negative control, and decoder-vector steering that beats an orthogonal random direction.


## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
